# 02 - Cleaning and merging (Transfermarkt dataset)

**Why this dataset:** football-data.org's free tier returned no market values and no contract dates (0 of 543 players), so market values come from the Transfermarkt-based dataset (`player-scores` on Kaggle), which also has minutes played, assists and international caps.

**Goal:** one row per Premier League player for the **2025/26 season**:
- target: the player's latest market value inside the season window
- features: age, position, minutes, goals and assists (per 90), international caps, contract years left

The notebook prints how many players are lost at each step ("funnel") so we can explain it in the presentation.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed"); PROCESSED.mkdir(parents=True, exist_ok=True)

LEAGUE = "GB1"                          # Premier League's code in Transfermarkt data
SEASON_START = pd.Timestamp("2025-07-01")
SEASON_END = pd.Timestamp("2026-06-30")
MIN_MINUTES = 450                       # ignore players with very little playing time (per-90 stats get noisy)

## 1. Check the league code

In [2]:
comps = pd.read_csv(RAW / "competitions.csv")
print(comps.columns.tolist())
comps[comps["competition_id"] == LEAGUE].T

['competition_id', 'competition_code', 'name', 'sub_type', 'type', 'country_id', 'country_name', 'domestic_league_code', 'confederation', 'total_clubs', 'url']


,29
competition_id,GB1
competition_code,premier-league
name,premier-league
sub_type,first_tier
type,domestic_league
country_id,189
country_name,England
domestic_league_code,GB1
confederation,europa
total_clubs,20.0


## 2. Appearances -> season totals per player

In [3]:
app = pd.read_csv(
    RAW / "appearances.csv",
    usecols=["player_id", "date", "competition_id", "goals", "assists",
             "minutes_played", "yellow_cards", "red_cards"],
)
app["date"] = pd.to_datetime(app["date"], errors="coerce")
print("all appearances in file:", len(app))

app = app[(app["competition_id"] == LEAGUE) & app["date"].between(SEASON_START, SEASON_END)]
assert len(app) > 0, "No Premier League appearances in the window - check LEAGUE code and dates"
print("PL appearances in 2025/26:", len(app), "| from", app["date"].min().date(), "to", app["date"].max().date())

all appearances in file: 1894350
PL appearances in 2025/26: 11492 | from 2025-08-15 to 2026-05-24


In [4]:
stats = (
    app.groupby("player_id")
       .agg(appearances=("date", "count"),
            minutes=("minutes_played", "sum"),
            goals=("goals", "sum"),
            assists=("assists", "sum"),
            yellow_cards=("yellow_cards", "sum"),
            red_cards=("red_cards", "sum"))
       .reset_index()
)
print("players with at least one PL appearance:", len(stats))
stats.sort_values("goals", ascending=False).head()

players with at least one PL appearance: 537


,player_id,appearances,minutes,goals,assists,yellow_cards,red_cards
221,418560,35,2959,27,8,2,0
424,739443,38,3287,22,1,7,0
343,583255,37,3205,17,4,7,0
124,324358,37,2854,16,3,4,0
231,429014,37,3113,15,4,1,0


## 3. Player profile (position, birth date, caps, contract)

In [5]:
players = pd.read_csv(
    RAW / "players.csv",
    usecols=["player_id", "name", "position", "sub_position", "date_of_birth",
             "international_caps", "contract_expiration_date", "current_club_name"],
)
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"], errors="coerce")
players["contract_expiration_date"] = pd.to_datetime(players["contract_expiration_date"], errors="coerce")
print(players.shape)
players.isna().mean().round(3)

(50149, 8)


player_id                   0.000
name                        0.000
date_of_birth               0.001
sub_position                0.012
position                    0.000
contract_expiration_date    0.370
international_caps          0.614
current_club_name           0.060
dtype: float64

## 4. Market value = the player's latest valuation inside the season window

`player_valuations.csv` is a history of valuations. For each player we take the most recent one dated within 2025/26. Players with no valuation in that window are dropped (and counted below).

In [6]:
val = pd.read_csv(RAW / "player_valuations.csv", usecols=["player_id", "date", "market_value_in_eur"])
val["date"] = pd.to_datetime(val["date"], errors="coerce")
val = val[val["date"].between(SEASON_START, SEASON_END)]
val = (
    val.sort_values("date")
       .groupby("player_id").tail(1)
       .rename(columns={"date": "valuation_date", "market_value_in_eur": "market_value"})
)
print("players with a valuation in the window:", len(val))
val["valuation_date"].describe()

players with a valuation in the window: 19280


count                         19280
mean     2026-05-04 16:32:59.253112
min             2025-07-04 00:00:00
25%             2026-05-21 00:00:00
50%             2026-05-28 00:00:00
75%             2026-06-03 00:00:00
max             2026-06-12 00:00:00
Name: valuation_date, dtype: object

## 5. Merge and build features

In [7]:
funnel = {}
funnel["PL players with appearances"] = len(stats)

df = stats.merge(players, on="player_id", how="left")
funnel["after joining profile"] = len(df)

df = df.merge(val, on="player_id", how="inner")
funnel["with a valuation in 2025/26"] = len(df)

df = df[df["minutes"] >= MIN_MINUTES]
funnel[f">= {MIN_MINUTES} minutes"] = len(df)

df["age"] = (df["valuation_date"] - df["date_of_birth"]).dt.days / 365.25
df["contract_years_left"] = ((df["contract_expiration_date"] - df["valuation_date"]).dt.days / 365.25).clip(lower=0)
df["goals_per90"] = df["goals"] / df["minutes"] * 90
df["assists_per90"] = df["assists"] / df["minutes"] * 90
df["international_caps"] = df["international_caps"].fillna(0)

df = df.dropna(subset=["age", "position", "market_value"])
df = df[df["market_value"] > 0]
funnel["final (no missing age/position/value)"] = len(df)

pd.Series(funnel, name="players").to_frame()

,players
PL players with appearances,537
after joining profile,537
with a valuation in 2025/26,522
>= 450 minutes,400
final (no missing age/position/value),400


### Sanity check

The most valuable players should look right (stars at the top). Missing values per column are shown after.

In [8]:
cols = ["name", "current_club_name", "position", "age", "minutes", "goals", "assists",
        "goals_per90", "assists_per90", "international_caps", "contract_years_left",
        "valuation_date", "market_value"]
df = df[["player_id"] + cols].reset_index(drop=True)
display(df.sort_values("market_value", ascending=False).head(10))
print(df.isna().sum()[lambda s: s > 0])

,player_id,name,current_club_name,position,age,minutes,goals,assists,goals_per90,assists_per90,international_caps,contract_years_left,valuation_date,market_value
177,418560,Erling Haaland,Manchester City,Attack,25.867214,2959,27,8,0.821223,0.243325,54.0,8.073922,2026-06-03,200000000
134,357662,Declan Rice,Arsenal FC,Midfield,27.383984,3099,4,7,0.116167,0.203291,77.0,2.075291,2026-06-03,120000000
189,433177,Bukayo Saka,Arsenal FC,Attack,24.741958,2226,7,7,0.283019,0.283019,54.0,4.073922,2026-06-03,110000000
237,495666,William Saliba,Arsenal FC,Defender,25.193703,2615,1,0,0.034417,0.000000,37.0,4.073922,2026-06-03,100000000
274,568177,Cole Palmer,Chelsea FC,Midfield,24.076660,1965,10,1,0.458015,0.045802,14.0,7.074606,2026-06-03,100000000
287,598577,Florian Wirtz,Liverpool FC,Midfield,23.085558,2390,5,4,0.188285,0.150628,45.0,4.073922,2026-06-03,100000000
200,451276,Dominik Szoboszlai,Liverpool FC,Midfield,25.604381,3233,6,7,0.167028,0.194865,65.0,2.075291,2026-06-03,100000000
326,687626,Moisés Caicedo,Chelsea FC,Midfield,24.583162,2802,3,1,0.096360,0.032120,65.0,7.074606,2026-06-03,100000000
291,607223,Rayan Cherki,Manchester City,Midfield,22.795346,1786,4,12,0.201568,0.604703,12.0,4.073922,2026-06-03,90000000
245,503743,Morgan Rogers,Aston Villa,Midfield,23.854894,3285,10,7,0.273973,0.191781,19.0,5.073238,2026-06-03,90000000


contract_years_left    16
dtype: int64


In [9]:
df.describe().T[["count", "mean", "min", "50%", "max"]].round(2)

,count,mean,min,50%,max
player_id,400.0,478149.4275,3333.0,450663.0,1230661.0
age,400.0,26.836242,17.760438,26.536619,40.410678
minutes,400.0,1830.455,452.0,1762.0,3420.0
goals,400.0,2.46,0.0,1.0,27.0
assists,400.0,1.885,0.0,1.0,21.0
goals_per90,400.0,0.118856,0.0,0.072556,0.821223
assists_per90,400.0,0.096751,0.0,0.076547,0.616037
international_caps,400.0,27.0675,0.0,16.5,151.0
contract_years_left,384.0,3.009468,0.073922,3.074606,8.073922
valuation_date,400,2026-06-02 21:18:00,2026-05-21 00:00:00,2026-06-03 00:00:00,2026-06-03 00:00:00


## 6. Save

In [10]:
out = PROCESSED / "PL_2025_26_model_data.csv"
df.to_csv(out, index=False)
print("saved", out, df.shape)

saved ../data/processed/PL_2025_26_model_data.csv (400, 14)


## Notes for the presentation

- The funnel above shows how many players survive each step.
- Contract years left is missing for some players; the model notebook fills those with the median.
- International caps is the latest snapshot in `players.csv`, not the value on the valuation date.
- The dataset snapshot ends in July 2026 (updates are paused), so this is a fixed 2025/26 sample.